# Predicting Next-Day Stock Price Direction

A walkthrough of the modelling pipeline: **data → features → train → evaluate**.

We predict whether a stock's close tomorrow will be higher than today's close.
This is a binary classification problem with a time-series twist: the split must
respect time so the test set is strictly in the future.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.make_dataset import load_market_data
from src.features import build_features, FEATURES, split_time
from src.train import build_models, train_and_evaluate
from src.evaluate import make_plots

In [ ]:
data = load_market_data()
data.head()

In [ ]:
features = build_features(data)
print(f"{len(features)} rows, {features['ticker'].nunique()} tickers")
print(f"features: {FEATURES}")
features[['date', 'ticker', 'close', 'ret_1', 'rsi_14', 'target']].head()

## Time-ordered split

The last 20% of *calendar dates* form the test set — no shuffle, no leakage.

In [ ]:
train, test = split_time(features)
print(f"train: {train['date'].min().date()} -> {train['date'].max().date()}  ({len(train)} rows)")
print(f"test : {test['date'].min().date()} -> {test['date'].max().date()}  ({len(test)} rows)")

## Train & compare models

We compare a majority-class baseline against logistic regression, a random
forest and gradient boosting.

In [ ]:
metrics, proba, importance = train_and_evaluate(features)
metrics.round(4)

## Interpretation

For this simulated market the direction is close to a coin flip, so every model
hovers near 50% accuracy — the **honest, expected result** for a near-efficient
market. The value here is the methodology: a leakage-free time split, a proper
baseline, and transparent metrics. On real data with genuine microstructure
signals the same pipeline would surface whatever edge exists.

In [ ]:
importance.head(10)

In [ ]:
from IPython.display import Image, display
for p in make_plots():
    display(Image(filename=str(p)))